# IOAI — 2024 Final Stage Ciphers (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/ciphered_lines.txt'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2024-final-stage-ciphers/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 암호 해독 — 베이스라인 (Szyfry, 빈도 매핑)

각 이모지(스티커)는 원래 한 글자로 치환된 것이다(글자당 최대 3개, 겹치지 않음). 이 베이스라인은 **단순 빈도
매핑** — 이모지 빈도와 (clear 코퍼스의) 글자 빈도를 순위로 맞춘다. 문맥을 안 쓰므로 정확도가 낮다(≈0.30).

`decipher_corpus(clear, ciphered)` → 해독된 줄들. 전체 원문/규칙은 Overview 탭 참고.

## 데이터 로드

In [ ]:
import numpy as np
from collections import Counter
corpus_clear    = [l.strip().lower()  for l in open("data/clear_lines.txt", encoding="utf-8")]
corpus_ciphered = [l.strip().lower()  for l in open("data/ciphered_lines.txt", encoding="utf-8")]
print("clear", len(corpus_clear), "| ciphered", len(corpus_ciphered))

## decipher_corpus — 빈도 매핑

In [ ]:
def decipher_corpus(clear_corpus, ciphered_corpus):
    chars = sorted(set("".join(clear_corpus)))
    syms  = sorted(set("".join(ciphered_corpus)))
    ccnt = Counter("".join(clear_corpus)); scnt = Counter("".join(ciphered_corpus))
    cuni = np.array([ccnt[c] for c in chars], float); euni = np.array([scnt[s] for s in syms], float)
    # 이모지(빈도 내림차순)를 글자(빈도 내림차순)에 누적질량으로 매칭
    eo = np.argsort(-euni); co = np.argsort(-cuni)
    ecum = np.cumsum(euni[eo])/euni.sum(); ccum = np.cumsum(cuni[co])/cuni.sum()
    m = {}; j = 0
    for k, e in enumerate(eo):
        while j < len(chars)-1 and ecum[k] > ccum[j]: j += 1
        m[syms[e]] = chars[co[j]]
    return ["".join(m.get(s, s) for s in line) for line in ciphered_corpus]

## 해독 → submission.txt

In [ ]:
deciphered = decipher_corpus(corpus_clear, corpus_ciphered)
with open("submission.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(deciphered))
print("saved submission.txt", len(deciphered), "lines")

문맥(bigram/trigram)을 쓰면 크게 개선된다 — 모범답안(trigram 언덕오르기) 참고.

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.txt']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)